In [1]:
DATA_PATH = r'input'

In [2]:
with open('events.txt') as f:
    event_names = tuple(line.strip() for line in f if line.strip())

event_names[:5]

FileNotFoundError: [Errno 2] No such file or directory: 'events.txt'

In [3]:
import pandas as pd
import glob
import os

def load_file(path):
    df = pd.read_csv(path)
    slash_cols = [c for c in df.columns if c not in ('userAgent', 'userIdentity.arn')]
    bad_slash = df[slash_cols].astype(str).apply(lambda col: col.str.contains('/', na=False)).any(axis=1)
    if bad_slash.any():
        raise ValueError(f"Unexpected '/' in non-userAgent fields in {path}: rows {df.index[bad_slash].tolist()}")
    bad_source = ~df['eventSource'].astype(str).str.endswith('.amazonaws.com')
    if bad_source.any():
        raise ValueError(f"Invalid eventSource values in {path}: {df.loc[bad_source, 'eventSource'].unique().tolist()}")
    last_col = df.columns[-1]
    if last_col == 'rank':
        df['type'] = 'ML Anomaly Detection: PYOD'
    elif last_col == 'outlier_kmeans_score':
        df['type'] = 'ML Anomaly Detection: K-means:'
        df['rank'] = '11'
    return df

files = glob.glob(os.path.join(DATA_PATH, '*.csv'))
signals_raw = pd.concat((load_file(f) for f in files), ignore_index=True)
signals_raw.head()

ValueError: No objects to concatenate

In [ ]:
from collections import Counter
from IPython.display import display

# corroborated: events detected by both pyod and kmeans
types_per_event = signals_raw.groupby('eventID')['type'].nunique()
duplicate_ids = types_per_event[types_per_event > 1].index
corroborated = (
    signals_raw[signals_raw['eventID'].isin(duplicate_ids)]
    .drop(columns=['outlier_kmeans_score', 'rank'], errors='ignore')
    .assign(type='Signal Fusion: Model Convergence')
    .drop_duplicates()
    .sort_values('eventID')
    .reset_index(drop=True)
)

# spec_based: events matching specification rules
spec_based = (
    signals_raw[signals_raw['eventName'].isin(event_names)]
    .drop(columns=['rank', 'outlier_kmeans_score'], errors='ignore')
    .assign(type='specification based detection')
    .drop_duplicates()
)

# signal_fusion: spec_based events also caught by ML
ml_ids = signals_raw[signals_raw['type'].isin(['ML Anomaly Detection: PYOD', 'ML Anomaly Detection: K-means:'])]['eventID']
signal_fusion = (
    spec_based[spec_based['eventID'].isin(ml_ids)]
    .drop(columns=['rank', 'outlier_kmeans_score'], errors='ignore')
    .copy()
    .drop_duplicates()
    .assign(**{'type': 'Signal Fusion: ML and Conventional Alert'})
)

# all_detections tuple
all_detections = tuple(
    (row['eventID'], row['type'])
    for df in [corroborated, spec_based, signal_fusion]
    for _, row in df[['eventID', 'type']].drop_duplicates().iterrows()
)

# signals_processed
signals_processed = pd.concat([
    signals_raw.drop(columns=['rank', 'outlier_kmeans_score'], errors='ignore'),
    corroborated,
    spec_based,
    signal_fusion
], ignore_index=True).drop_duplicates()

# per-ARN breakdown with totals
pivot = signals_processed.pivot_table(
    index='userIdentity.arn',
    columns='type',
    aggfunc='size',
    fill_value=0
)
pd.concat([pivot, pivot.sum().rename('Total').to_frame().T])

type,ML Anomaly Detection: K-means:,ML Anomaly Detection: PYOD,Signal Fusion: ML and Conventional Alert,Signal Fusion: Model Convergence,specification based detection
arn:aws:iam::123456789012:root,127,127,35,127,35
arn:aws:sts::123456789012:assumed-role/flowlogsRole/vpc-flow-logging+123456789012,4,4,0,4,0
Total,131,131,35,131,35


In [ ]:
import numpy as np

type_weights = {
    'ML Anomaly Detection: PYOD': 1,
    'ML Anomaly Detection: K-means:': 1,
    'specification based detection': 1,
    'Signal Fusion: Model Convergence': 2,
    'Signal Fusion: ML and Conventional Alert': 2,
}

def arn_score(types):
    counts = types.value_counts()
    diversity = sum(type_weights.get(t, 1) * np.log1p(c) for t, c in counts.items())
    volume = np.log1p(len(types))
    return diversity * volume

atomic_weight = (
    signals_processed
    .groupby('userIdentity.arn')['type']
    .apply(arn_score)
    .rename('atomic weight')
    .round(1)
)

k = atomic_weight.max() / 3
scores = (100 * np.tanh(np.exp(atomic_weight / k) - 1)).rename('score').round().astype(int)

atomic_mass = signals_processed.groupby('userIdentity.arn').size().rename('atomic mass')

pd.concat([atomic_weight, atomic_mass, scores], axis=1).sort_values('score', ascending=False)

,atomic weight,atomic mass,score
userIdentity.arn,,,
arn:aws:iam::123456789012:root,184.4,451,100
arn:aws:sts::123456789012:assumed-role/flowlogsRole/vpc-flow-logging+123456789012,16.5,12,30
